## Cell type marker genes identification


In [ ]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    # library(Libra)
    library(harmony)
    library(ComplexHeatmap)
    library(RColorBrewer)
    })
# set large memory  
options(future.globals.maxSize = 122*1024^3)

In [ ]:
## set working dir
dir = "/home/kibr/Work/Vascular_atlas"
setwd(dir)

In [ ]:
## Reloading 
# Convert h5ad to h5seurat
# h5ad_file="./Results/Revision_2/clean_object_revision_03032026.h5ad"
# clean = schard::h5ad2seurat(h5ad_file)/
clean = readRDS("./Results/Revision_2/clean_object_vascular_cell_types.rds")
print(clean)

In [ ]:
## Upodating the brain region annotation
region_anno = read.csv("./Data/region_update.csv")
print(region_anno)

meta = clean@meta.data
ID = match(meta$regionID, region_anno$regionID)
meta$region_abb = region_anno$Final_abb[ID]
meta$region_name = region_anno$Region_layer_1[ID]
meta$region_layer = region_anno$Region_layer_2[ID]

table(meta$region_abb)
table(meta$region_name)
table(meta$region_layer)

clean@meta.data = meta

In [ ]:
individual = "UW_7029"
meta = clean@meta.data
meta_sub = meta[meta$individualID == individual,]
table(meta_sub$Cell_class, meta_sub$region_abb)

In [ ]:
table(clean$Cell_type, clean$individualID)

## QC check up
## Figure 1

In [ ]:
## organizing the meta data 
meta = clean@meta.data
meta$cell.class = meta$Cell_class
table(meta$cell.class)

clean@meta.data = meta

In [ ]:
# plot
library(tidyverse)
library(viridis)
library(hrbrthemes)

meta <- clean@meta.data
p <- list()
for (i in 1:length(unique(meta$cell.class))){
   p[[i]] <- meta[meta$cell.class == unique(meta$cell.class)[i],]%>%
            ggplot( aes(x=nFeature_RNA, color=cell.class, fill=cell.class)) +
              geom_histogram() +
              scale_fill_viridis(discrete=TRUE) +
              scale_color_viridis(discrete=TRUE) +
              theme_ipsum() +
              theme(
                legend.position="none",
                panel.spacing = unit(0.1, "lines"),
                strip.text.x = element_text(size = 8)
              ) +
              xlab("Number of feature") +
              ylab("") +
              ggtitle(unique(meta$cell.class)[i]) 
}

In [ ]:
#######################################################
###----- Figure_1B: Object UMAP---------###
#######################################################
col1=c('#F06719','#08415C','#23767C','#155a66','#DC143C',"#00BFC4","#fcbe05","#A26DC2","#5b844d","#0072B2")
names(col1)=c('Astrocyte','Neuron','Ependymal_Cell','Epithelial_Cell','Microglia_Macrophage_T','Oligodendrocyte',"Endothelial","Mural_Cell","Fibroblast","OPC")

umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))
## ploting
p1 <- DimPlot(clean, label = F, repel = TRUE, 
                reduction = "umap.harmony.denoised",group.by = "cell.class",cols=col1,raster = T,pt.size = 5,raster.dpi=c(3200,3200)) + umap_theme+ggtitle("Cell_class")+NoLegend()
p2 <- DimPlot(clean, label = T, repel = TRUE, 
                reduction = "umap.harmony.denoised",group.by = "cell.class",cols=col1,raster = T,pt.size = 2) + umap_theme+ggtitle("Cell_class")
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1B_cell_class_umap.pdf",
#     patchwork::wrap_plots(p1, ncol = 1),
#       scale = 1, width = 9, height = 9)
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1B_cell_class_umap_legend.pdf",
#     patchwork::wrap_plots(p2, ncol = 1),
#       scale = 1, width = 11, height = 9)

# options(repr.plot.width = 11,repr.plot.height = 9)
# p1
# p2

## Cell type proportion plot

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)

In [ ]:
# ── 0. Load region metadata ───────────────────────────────────────────────────
region_meta <- read.csv("./Data/region_update.csv", stringsAsFactors = FALSE) %>%
  mutate(
    RegionID = str_trim(regionID),
    Region_layer_1    = str_to_title(str_trim(Region_layer_1)),
    Region_layer_2 = str_trim(Region_layer_2),
    Region_abb = str_trim(Final_abb)
  ) %>%
  select(RegionID, Region_layer_1, Region_layer_2, Region_abb)
  # region_meta

In [ ]:
## get meta data to work on
meta <- clean@meta.data
meta$region_name = str_to_title(str_trim(meta$region_name))
meta$brain_region <- paste(meta$regionID,meta$region_name, sep = "_")
# table(meta$brain_region)

In [ ]:
# ── 1. Get the count table ────────────────────────────────────────────────────
counts <- ddply(meta, .(meta$Cell_class, meta$brain_region), nrow)
colnames(counts) <- c("cell_class", "brain_region", "freq")

## addint the region_abb to the another column
df = unique(meta[,c("brain_region","region_abb")])
rownames(df) = df$brain_region
counts$region_abb = df[counts$brain_region, "region_abb"]

# ── 2. Parse numeric order and abbreviation from brain_region (e.g. "1_HIP") ─
counts$order           <- as.numeric(str_split_fixed(counts$brain_region, "_", n = 2)[, 1])
counts$brain_region_plot <- str_split_fixed(counts$brain_region, "_", n = 2)[, 2]

In [ ]:
# ── 3. Join region_layer_2 from region CSV via Final_abb ─────────────────────
counts <- counts %>%
  left_join(region_meta, by = c("brain_region_plot" = "Region_layer_1"))

# ── 4. Define region_layer_2 display order ───────────────────────────────────
layer2_order <- c(
  "Cortex",
  "Limbic",
  "White Matter Tracts",
  "Brainstem",
  "Cerebellum",
  "Watershed",
  "Major Vessel",
  "Olfactory",
  "Barrier"
)
# ── 5. Sort: first by layer2 group order, then by numeric region order within ─
counts$layer2_order_idx <- match(counts$Region_layer_2, layer2_order)
counts <- counts[order(counts$layer2_order_idx, counts$order), ]

# ── 6. Set brain_region_plot as ordered factor (preserves sort in ggplot) ─────
counts$brain_region_plot <- factor(
  counts$brain_region_plot,
  levels = unique(counts$brain_region_plot)
)
## modify the brain_region_plot to include the region_abb in parentheses
counts$brain_region_plot_2 <- paste(counts$brain_region_plot," (",counts$region_abb,")", sep = "")
counts$brain_region_plot_2 = factor(counts$brain_region_plot_2, levels = unique(counts$brain_region_plot_2))

# ── 7. Set Region_layer_2 as ordered factor (for facet / fill ordering) ───────
counts$Region_layer_2 <- factor(
  counts$Region_layer_2,
  levels = layer2_order
)
# counts

In [ ]:
region_levels         <- unique(counts$brain_region_plot)
counts$brain_region_plot <- factor(counts$brain_region_plot, levels = region_levels)
counts$Region_layer_2    <- factor(counts$Region_layer_2,    levels = layer2_order)

In [ ]:
# ── 3. Color maps ─────────────────────────────────────────────────────────────
col1 <- c(
  Astrocyte              = "#F06719", Neuron              = "#08415C",
  Ependymal_Cell         = "#23767C", Epithelial_Cell     = "#155a66",
  Microglia_Macrophage_T = "#DC143C", Oligodendrocyte     = "#00BFC4",
  Endothelial            = "#fcbe05", Mural_Cell          = "#A26DC2",
  Fibroblast             = "#5b844d", OPC                 = "#0072B2"
)
counts$cell_class <- factor(counts$cell_class)
 
region_color_map <- c(
    "Cortex" =  "#009E73",
    "Brainstem" = "#D55E00",
    "Cerebellum" = "#046299",
    "Limbic" = "#03B8D8",
    "Watershed" = "#E69F00",
    "White Matter Tracts" = "#CC79A7",
    "Olfactory" = "#999999",
    "Barrier" = "#9C0AE0",
    # "Subcortical" = "#915330",
    "Major Vessel" = "#FF0000"
)

# ── 4. Lookup: region → layer2 color ─────────────────────────────────────────
rl2_lookup <- counts %>%
  select(brain_region_plot, Region_layer_2) %>%
  distinct() %>%
  mutate(brain_region_plot = as.character(brain_region_plot),
         Region_layer_2    = as.character(Region_layer_2))
 
tick_colors <- region_color_map[
  rl2_lookup$Region_layer_2[match(region_levels, rl2_lookup$brain_region_plot)]
]

# ── 5. Build dendrogram segments ─────────────────────────────────────────────
# Dendrogram is drawn top-down, then y-axis is reversed so it hangs below bars.
# y = 0  → top (connects to bar x-axis)
# y = 1  → horizontal grouping bar
# y = 2  → stem tip / label
region_x <- setNames(seq_along(region_levels), region_levels)
 
dend_segs  <- list()
dend_label <- list()
 
for (l2 in layer2_order) {
  members <- rl2_lookup %>% filter(Region_layer_2 == l2) %>% pull(brain_region_plot)
  if (length(members) == 0) next
 
  xs    <- region_x[members]
  x_min <- min(xs);  x_max <- max(xs);  x_mid <- (x_min + x_max) / 2
  col   <- region_color_map[[l2]]
 
  # Vertical ticks: top (y=0) → horizontal bar (y=1)
  for (xi in xs) {
    dend_segs[[length(dend_segs) + 1]] <- data.frame(
      x = xi, xend = xi, y = 0, yend = 1, col = col, stringsAsFactors = FALSE)
  }
  # Horizontal bar at y=1
  dend_segs[[length(dend_segs) + 1]] <- data.frame(
    x = x_min, xend = x_max, y = 1, yend = 1, col = col, stringsAsFactors = FALSE)
  # Vertical stem: y=1 → y=2
  dend_segs[[length(dend_segs) + 1]] <- data.frame(
    x = x_mid, xend = x_mid, y = 1, yend = 2, col = col, stringsAsFactors = FALSE)
 
  # Label at y=2 (will appear below stem after y-axis reversal)
  dend_label[[length(dend_label) + 1]] <- data.frame(
    x = x_mid, y = 2.08, label = l2, col = col, stringsAsFactors = FALSE)
}
 
dend_segs_df  <- bind_rows(dend_segs)
dend_label_df <- bind_rows(dend_label)
n_regions     <- length(region_levels)
 
# ── 6. Dendrogram panel (y reversed: y=0 at top, labels hang downward) ───────
p_dend <- ggplot() +
  geom_segment(data = dend_segs_df,
               aes(x = x, xend = xend, y = y, yend = yend, colour = I(col)),
               linewidth = 0.65) +
  geom_text(data = dend_label_df,
            aes(x = x, y = y, label = label, colour = I(col)),
            angle = 90, hjust = 1, vjust = 0.5,
            size = 2.7, fontface = "bold") +
  scale_x_continuous(limits = c(0.5, n_regions + 0.5), expand = c(0, 0)) +
  scale_y_reverse(limits = c(6, 0), expand = c(0, 0)) +
  theme_void() +
  theme(plot.margin = margin(0, 2, 2, 2))
 
# ── 7. Bar plot panel (no x-axis labels — dendrogram carries them) ────────────
p_bar <- ggplot(counts, aes(x = brain_region_plot_2, y = freq, fill = cell_class)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = col1, name = "Cell class") +
  scale_x_discrete(expand = c(0, 0.5)) +
  labs(x = NULL, y = "Cell count") +
  theme_bw(base_size = 10) +
  theme(
    axis.text.x        = element_text(angle = 90, hjust = 1, vjust = 0.5,
                                      size = 7, colour = tick_colors),
    axis.ticks.x       = element_line(colour = tick_colors),
    panel.grid.major.x = element_blank(),
    panel.grid.minor   = element_blank(),
    legend.position    = "bottom",
    legend.key.size    = unit(0.4, "cm"),
    plot.margin        = margin(5, 2, 0, 2)
  )
# ── 8. Stack: bar on top, dendrogram on bottom ────────────────────────────────
final_plot <- p_bar / p_dend +
  plot_layout(heights = c(4, 1.8))

ggsave("./Results/Revision_2/Figures/Figure_1X_cell_counts_with_dendrogram.pdf", plot = final_plot,
       width = 8, height = 5.2, bg = "white")
message("Saved: cell_counts_with_dendrogram.pdf")
final_plot

## For Supplementary figure 1: Proportional plot with sum=100%

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)
## get the count table
meta = clean@meta.data
meta$brain_region <- paste(meta$regionID,meta$region_abb, sep = "_")

counts <- ddply(meta, .(meta$Cell_class, meta$brain_region), nrow)
colnames(counts) <- c("cell_class","brain_region","freq")

## reorder the brain region based on the number code
counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)
## get the count table
counts <- ddply(meta, .(meta$Cell_class, meta$brain_region), nrow)
colnames(counts) <- c("cell_class","brain_region","freq")

## reorder the brain region based on the number code
counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])
counts <- counts[order(counts$cell_class),]
counts <- counts[order(counts$order),]

## making sure corrected order
counts$brain_region_plot = str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,2]
counts$brain_region_plot<- factor(counts$brain_region_plot, levels=unique(counts$brain_region_plot))


counts$cell_class <- factor(counts$cell_class)
## colors
col1=c('#F06719','#08415C','#23767C','#155a66','#DC143C',"#00BFC4","#fcbe05","#A26DC2","#5b844d","#0072B2")
names(col1)=c('Astrocyte','Neuron','Ependymal_Cell','Epithelial_Cell','Microglia_Macrophage_T','Oligodendrocyte',"Endothelial","Mural_Cell","Fibroblast","OPC")
counts$color <- col1[counts$cell_class]
# counts

In [ ]:
# plotting the cell types
p3 <- ggplot(counts, aes(fill=cell_class, y=freq, x=brain_region_plot)) + 
        geom_bar(position="stack", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        scale_fill_manual(values=col1) +
        xlab("")

p4 <- ggplot(counts, aes(fill=cell_class, y=freq, x=brain_region_plot)) + 
        geom_bar(position="fill", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 15),
              axis.text.y = element_text(size = 15),
              axis.title.y = element_text(size = 15)) +
        scale_fill_manual(values=col1) +
        xlab("") +
        ylab("Frequency")
options(repr.plot.width = 12, repr.plot.height = 8, repr.plot.res = 100) # set the resolution
p3
p4
ggsave(filename = "./Results/Revision_2/Figures/Figure_1E_Cell_class_region_proportion.pdf", 
patchwork::wrap_plots(p4, ncol = 1),
scale = 1, width = 12, height = 5)

## Plot the proportion of individual contributing to each cell class in each region

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)

# ----------------------------
# 1) Count per cell_class x brain_region x individualID
# ----------------------------'
meta = clean@meta.data
meta$brain_region <- paste(meta$regionID,meta$region_abb, sep = "_")

counts_ind <- ddply(meta, .(meta$Cell_class, meta$brain_region, meta$individualID), nrow)
colnames(counts_ind) <- c("cell_class", "brain_region", "individualID", "freq")

# ----------------------------
# 2) Reorder brain_region same as before
# ----------------------------
counts_ind$order <- as.numeric(str_split_fixed(counts_ind$brain_region, pattern = "_", n = 2)[, 1])
counts_ind <- counts_ind[order(counts_ind$cell_class), ]
counts_ind <- counts_ind[order(counts_ind$order), ]

counts_ind$brain_region_plot <- str_split_fixed(counts_ind$brain_region, pattern = "_", n = 2)[, 2]
counts_ind$brain_region_plot <- factor(counts_ind$brain_region_plot, levels = unique(counts_ind$brain_region_plot))
counts_ind$cell_class <- factor(counts_ind$cell_class)

# ----------------------------
# 3) Compute proportion within each cell_class x brain_region
# ----------------------------
counts_ind <- ddply(counts_ind, .(cell_class, brain_region_plot), transform,
                    prop = freq / sum(freq))

# ----------------------------
# 4) Colors per individualID (adjust to your actual IDs)
# ----------------------------
ind_colors <- c(
    "LB_4008" = "#0072B2",
    "LB_9770" = "#E69F00",
    "Stanford_12052023" = "#009E73",
    "UW_7029" = "#D55E00"
)

# ----------------------------
# 5) Plot: one facet per cell_class, x = brain_region, fill = individualID
# ----------------------------
p_ind <- ggplot(counts_ind, aes(x = brain_region_plot, y = prop, fill = individualID)) +
    geom_bar(position = "fill", stat = "identity") +
    facet_wrap(~ cell_class, ncol = 2) +
    scale_fill_manual(values = ind_colors) +
    theme_minimal() +
    theme(
        axis.text.x  = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 14),
        axis.text.y  = element_text(size = 14),
        axis.title.y = element_text(size = 14),
        strip.text   = element_text(size = 14, face = "bold"),
        legend.title = element_text(size = 18)
    ) +
    xlab("") +
    ylab("Proportion of individualID")

options(repr.plot.width = 16, repr.plot.height = 12, repr.plot.res = 100)
p_ind

ggsave(
    filename = "./Results/Revision_2/Figures/Sup_Figure_1d_individualID_proportion_by_region_cellclass.pdf",
    patchwork::wrap_plots(p_ind, ncol = 1),
    scale = 1, width = 16, height = 12
)

In [ ]:
individual_ids <- unique(counts_ind$individualID)
## length = 4   
ind_colors <- setNames(
    colorRampPalette(c("#1f77b4","#ff7f0e","#2ca02c","#d62728",))(length(individual_ids)),
    individual_ids
)

### Chi-sq test to check cell type enrichment

In [ ]:
### try chisq contigency table
meta <- clean@meta.data
# meta$brain_region <- paste(meta$regionID,meta$region_abb,sep = "_")

obs_mat = as.matrix(table(meta$region_abb,meta$Cell_class))

In [ ]:
# run chisq test
chisq_res <- chisq.test(obs_mat)
exp_mat = chisq_res$expected

# get the log1p(mat)
res = (obs_mat/exp_mat)+1
res = log(res)
res = res[order(as.numeric(str_split_fixed(rownames(res),pattern = "_",n = 2)[,1])),]

In [ ]:
## plotting
library(ComplexHeatmap)
library(colorRamp2)

#options(repr.plot.width = 10, repr.plot.height = 12, repr.plot.res = 100) # set the resolution
p = Heatmap(res,col = colorRamp2(c(0,1.5,3),c("grey95","red","red4")),cluster_rows = T,cluster_columns = F)

pdf("./Results/Revision_2/Figures/Figure_S1_Cell_state_enrichment_cluster_ALL.pdf",width = 5,height = 11)
p
dev.off()


## Redo the normalization for cell type marker gene identification
#### check if the data is clean enough

In [ ]:
## set cell identity
DefaultAssay(clean) <- "decontX"
clean
Idents(clean) <- "Cell_class"
table(Idents(clean)) 
## normalization
clean <- NormalizeData(clean)
clean <- JoinLayers(clean)

In [ ]:
#### code for identifying the cell type makers ####
# the parameters are based on https://www.cell.com/cell/fulltext/S0092-8674(23)00973-X
table(Idents(clean))
# clean <- PrepSCTFindMarkers(clean)
# running differential analysis
all_markers_major <- FindAllMarkers(object = clean,verbose = T,min.pct = 0.25,logfc.threshold = 0.25
#,test.use = "MAST" #might use MAST model
) 
write.csv(all_markers_major,file = "./Results/Revision_2/DEG/All_cellclass_markers_Wilcoxon_0304.csv")

### Load the cell type DEG results

In [ ]:
######################################################################
###########-- check cell type marker genes on each cluster --#########
######################################################################
library(viridis)
Idents(clean) <- "Cell_class"
Idents(clean) <- factor(clean@active.ident, sort(levels(clean@active.ident),decreasing = T))

## set assay to RNA
DefaultAssay(clean) <- "decontX"

p1 <-  DotPlot(clean,cols = viridis(3, option = "magma"),
             features = c("SLC1A2","RYR3","AQP4","GFAP",#Astrocytes
                            "FLT1","MECOM","ATP10A",  # Endothelial
                            "CFAP54","CFAP46","HTR2C","ABCA4", # Ependymal_Epithelial
                            "CEMIP","BICC1","LAMA2", #Fibroblast
                            "C3","RYR1","DOCK2",# Microglia_Macrophage_T
                            "CARMN","MYH11","RGS5", #Mural cell
                            "SYT1","ROBO2","CNTNAP2","GAD1", # Neuron
                            "MOBP","ST18", "SLC24A2",#OLI
                            "TNR","DSCAM","LHFPL3" # OPC
                                )) + RotatedAxis() +
                                    theme(axis.title = element_blank(),legend.position = "top")
                                    
ggsave(filename = "./Results/Revision_2/Figures/Figure_1C_cellclass_dot_plot.pdf",
   patchwork::wrap_plots(p1, ncol = 1),
     scale = 1, width = 8, height = 4)

# p1

## Save copy in h5ad format for other python based analysis

In [ ]:
## check if python is available and check which version is under using
## python packages ## wired issue
# reticulate::use_python("/oak/stanford/projects/kibr/Reorganizing/Projects/Andi/software/miniforge3/envs/sc/bin")
reticulate::py_discover_config()
#py_install("numpy")
reticulate::py_module_available("numpy")
reticulate::py_module_available("anndata")

In [ ]:
## check before subsetting
clean
options(repr.plot.width= 10)
DimPlot(clean, reduction = "umapharmony_",group.by = "Cell_type")

In [ ]:
DefaultAssay(clean) <- "RNA"
#integrated <- JoinLayers(integrated)
clean[["RNA"]] <- as(object = clean[["RNA"]], Class = "Assay")
clean[["RNA"]] # check the version of the seurat assay
sceasy::convertFormat(clean, from="seurat", to="anndata",
                       outFile='./Results/03.object_class_0620.h5ad')

## Plot more QC matrics

In [ ]:
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))
p0 <- DimPlot(clean, reduction = "umapharmony_",group.by = "cell.class",label = T,raster = T,pt.size = 2)+umap_theme+NoLegend()
p1 <- DimPlot(clean, reduction = "umapharmony_",group.by = "individualID",raster = T,pt.size = 2)+umap_theme+NoLegend()
p2 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sampleID",raster = T,pt.size = 2)+umap_theme+NoLegend()
p3 <- DimPlot(clean, reduction = "umapharmony_",group.by = "brain_region",raster = T,pt.size = 2)+umap_theme+NoLegend()
p4 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sex",raster = T,pt.size = 2)+umap_theme+NoLegend()
p5 <- FeaturePlot(clean, reduction = "umapharmony_",features = "ageatdeath",cols = brewer.pal(n = 8, name = 'Blues'),raster = T,pt.size = 2)+umap_theme+NoLegend()

In [ ]:
# ggsave(filename = "./Results/Revision_2/Figures/Figure_S1_clean_object_QC.pdf",
#     patchwork::wrap_plots(p0,p1,p2,p3,p4,p5, ncol = 3),
#       scale = 1, width = 21, height = 14)

In [ ]:
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))
p0 <- DimPlot(clean, reduction = "umapharmony_",group.by = "cell.class",label = T,raster = T,pt.size = 2)+umap_theme+NoLegend()
p1 <- DimPlot(clean, reduction = "umapharmony_",group.by = "individualID",raster = T,pt.size = 2)+umap_theme
p2 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sampleID",raster = T,pt.size = 2)+umap_theme+NoLegend()
p3 <- DimPlot(clean, reduction = "umapharmony_",group.by = "brain_region",raster = T,pt.size = 2)+umap_theme+NoLegend()
p4 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sex",raster = T,pt.size = 2)+umap_theme
p5 <- FeaturePlot(clean, reduction = "umapharmony_",features = "ageatdeath",cols = brewer.pal(n = 8, name = 'Blues'),raster = T,pt.size = 2)+umap_theme

In [ ]:
# ggsave(filename = "./Results/Revision_2/Figures/Figure_S1_clean_object_metadata_legend.pdf",
#     patchwork::wrap_plots(p0,p1,p2,p3,p4,p5, ncol = 3),
#       scale = 1, width = 24, height = 14)

## Draw ComplexHeatmap to show donor by region matrix of number of cells

In [ ]:
### Draw ComplexHeatmap to show donor by region matrix of number of cells
meta = clean@meta.data
mat = as.matrix(table(meta$individualID,meta$region_abb))
# mat

In [ ]:
## Provide the color code
meta <- clean@meta.data
df <- unique(meta[,c("region_abb","region_layer",'regionID')])

df = as.data.frame(df)

df$region_color <- NA
df[df$region_layer == "Cortex",]$region_color = "#009E73"
df[df$region_layer == "Brainstem",]$region_color = "#D55E00"
df[df$region_layer == "Cerebellum",]$region_color = "#046299"
df[df$region_layer == "Limbic",]$region_color = "#03B8D8"
df[df$region_layer == "Watershed",]$region_color = "#E69F00"
df[df$region_layer == "White Matter Tracts",]$region_color = "#CC79A7"
df[df$region_layer == "Olfactory",]$region_color = "#999999"
df[df$region_layer == "Barrier",]$region_color = "#9C0AE0"
# df[df$region_layer == "Subcortical",]$region_color = "#915330"
df[df$region_layer == "Major Vessel",]$region_color = "#FF0000"

# ## Assign region color
region_color <- df$region_color
names(region_color) <-df$region_abb

mat = mat[,df[order(as.numeric(df$regionID)),]$region_abb]

In [ ]:
library(colorRamp2)
# annotation for heatmap
hl <- HeatmapAnnotation(
  region = colnames(mat),
#   avg_density = avg_vec,
  col = list(
    region = region_color
  ),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(mat),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(mat))
)
# Main heatmap
ht1 <- Heatmap(
  mat,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = FALSE,
  show_heatmap_legend = TRUE,
  use_raster = TRUE,
  col = colorRamp2(c(0, 500, 2000, 5000, 15000), 
           c("#FFFFFF", "#C6DBEF", "#6BAED6", "#2171B5", "#08306B"))
  # cell_fun = function(j, i, x, y, w, h, fill) {
  #   if (sig_mat[i, j] < 0.05) {
  #     grid.text("*", x, y, gp = gpar(fontsize = 20, col = "black"), vjust = "center")
  #   }
  # }
)

In [ ]:
pdf(file = "./Results/Revision_2/Figures/Sup_Figure_1X_number_of_cells_per_region.pdf", width = 13, height = 3)
ht1
dev.off()
ht1

## Draw ComplexHeatmap to show donor by region matrix of total nCount_RNA

In [ ]:
meta = clean@meta.data

# Sum nCount_RNA per donor per region instead of counting cells
mat = as.matrix(tapply(meta$nCount_RNA, list(meta$individualID, meta$region_abb), sum))
mat[is.na(mat)] = 0  # Replace NA with 0 for donor-region combinations with no cells
# mat

# Log10 transform (add 1 to avoid log(0))
mat_log = log10(mat + 1)
# mat_log

## Provide the color code
meta <- clean@meta.data
df <- unique(meta[,c("region_abb","region_layer",'regionID')])
df = as.data.frame(df)
df$region_color <- NA
df[df$region_layer == "Cortex",]$region_color = "#009E73"
df[df$region_layer == "Brainstem",]$region_color = "#D55E00"
df[df$region_layer == "Cerebellum",]$region_color = "#046299"
df[df$region_layer == "Limbic",]$region_color = "#03B8D8"
df[df$region_layer == "Watershed",]$region_color = "#E69F00"
df[df$region_layer == "White Matter Tracts",]$region_color = "#CC79A7"
df[df$region_layer == "Olfactory",]$region_color = "#999999"
df[df$region_layer == "Barrier",]$region_color = "#9C0AE0"
# df[df$region_layer == "Subcortical",]$region_color = "#915330"
df[df$region_layer == "Major Vessel",]$region_color = "#FF0000"

## Assign region color
region_color <- df$region_color
names(region_color) <- df$region_abb
mat_log = mat_log[, df[order(as.numeric(df$regionID)),]$region_abb]
mat_log

# Annotation for heatmap
hl <- HeatmapAnnotation(
  region = colnames(mat_log),
  col = list(
    region = region_color
  ),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(mat_log),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(mat_log))
)

# Compute color scale breaks based on data range
max_val = max(mat_log, na.rm = TRUE)
max_log = ceiling(max(mat_log, na.rm = TRUE))
# Main heatmap
ht2 <- Heatmap(
  mat_log,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = FALSE,
  show_heatmap_legend = TRUE,
  use_raster = FALSE,
  name = "log10_nCount_RNA",          # <-- no parentheses or special characters
    col = colorRamp2(
    seq(0, max_log, length.out = 5),
    c("#FFFFFF", "#C6DBEF", "#6BAED6", "#2171B5", "#08306B")
    )
)

In [ ]:
pdf(file = "./Results/Revision_2/Figures/Sup_Figure_1X_number_of_reads_per_sample.pdf", width = 13, height = 3)
ht2
dev.off()

## Number of replicates per region per donor

In [ ]:
meta <- clean@meta.data

# Count distinct samples (replicates) per donor per region
mat <- as.matrix(tapply(meta$sampleID, list(meta$individualID, meta$region_abb), function(x) length(unique(x))))
mat[is.na(mat)] <- 0

# NO log transform needed — we want raw counts
# But optionally you can keep it if counts vary widely:
# mat_log <- log10(mat + 1)

## Provide the color code
df <- unique(meta[, c("region_abb", "region_layer", "regionID")])
df <- as.data.frame(df)
df$region_color <- NA
df[df$region_layer == "Cortex",]$region_color        <- "#009E73"
df[df$region_layer == "Brainstem",]$region_color      <- "#D55E00"
df[df$region_layer == "Cerebellum",]$region_color     <- "#046299"
df[df$region_layer == "Limbic",]$region_color         <- "#03B8D8"
df[df$region_layer == "Watershed",]$region_color      <- "#E69F00"
df[df$region_layer == "White Matter Tracts",]$region_color <- "#CC79A7"
df[df$region_layer == "Olfactory",]$region_color      <- "#999999"
df[df$region_layer == "Barrier",]$region_color        <- "#9C0AE0"
# df[df$region_layer == "Subcortical",]$region_color    <- "#915330"
df[df$region_layer == "Major Vessel",]$region_color   <- "#FF0000"

## Assign region color
region_color <- df$region_color
names(region_color) <- df$region_abb

# Order columns by regionID
mat <- mat[, df[order(as.numeric(df$regionID)),]$region_abb]

# Top annotation: region color bar
hl <- HeatmapAnnotation(
  region = colnames(mat),
  col = list(region = region_color),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(mat),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(mat))
)

# Color scale: white = 0 samples, dark blue = max samples
max_count <- max(mat, na.rm = TRUE)

ht3 <- Heatmap(
  mat,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = FALSE,       # show donor IDs on rows
  show_heatmap_legend = TRUE,
  use_raster = FALSE,
  name = "n_samples",
  col = colorRamp2(
    seq(0, max_count, length.out = 3),
    c("#FFFFFF", "#2171B5", "#08306B")
  )
)


In [ ]:
pdf(file = "./Results/Revision_2/Figures/Sup_Figure_1X_number_of_replicates_per_donor.pdf", width = 13, height = 3)
ht3
dev.off()

## Saving object

In [ ]:
## check and save the clean object for future use
clean
saveRDS(clean,file = "./Results/Revision_2/clean_object_vascular_cell_types_04262026.rds")

## Plotting the expression of the marker genes on each cell type.

In [ ]:
table(clean$Cell_class)
c_oi = c("Endothelial","Mural_Cell",'Fibroblast')

clean = subset(clean, subset = clean$Cell_class %in% c_oi)
clean

In [ ]:
library(tidyverse)
library(viridis)
library(hrbrthemes)

table(clean$Cell_type)
Idents(clean) = "Cell_type"
my_order <- c("Large_Artery","Arterial", "Capillary", "Venous","Fenestrated_Capillary","EndoMT",
                "Pericyte","SMC_1","SMC_2","SMC_3",
                "Fib_1","Fib_2","Fib_3","Fib_4","Fib_5","Fib_6")

clean$Cell_type <- factor(clean$Cell_type, levels = rev(my_order))

genes_oi = c("PCDH9",'GPM6A',"IL1RAPL1","PPP2R2B","MYH11","TAGLN","FLT1","CARMN","CEMIP","ROBO2","MOBP","SLC1A2","CLDN5","CLDN11","CTNNA3","CADM2","BMX","GJA5")  

# ## Before decontX
# DefaultAssay(clean) = "RNA"
# # clean = NormalizeData(clean)
# p1 = DotPlot(clean,features = genes_oi,cols = viridis(3, option = "magma"),group.by = "Cell_type",scale = FALSE) +
#             RotatedAxis() + theme(axis.title = element_blank()) + ggtitle("Before decontX")


## After decontX
DefaultAssay(clean) = "decontX"
clean@assays$decontX$counts = round(LayerData(clean, assay = "decontX", layer = "counts"))

clean = NormalizeData(clean)
## plot the dot plot with color white to dark red
p2 = DotPlot(clean,features = genes_oi,cols = c("white","darkred"),group.by = "Cell_type",scale = FALSE,assay = "decontX",) +
            RotatedAxis() + theme(axis.title = element_blank()) + ggtitle("After decontX")


In [ ]:
## get the subsetted data
meta = clean@meta.data
og_meta = read.csv("./Results/original_meta.csv", stringsAsFactors = FALSE)

clean_sub = clean[,rownames(meta) %in% og_meta$X]
clean_sub

In [ ]:
table(clean_sub$Cell_type)
Idents(clean_sub) = "Cell_type"
my_order <- c("Large_Artery","Arterial", "Capillary", "Venous","Fenestrated_Capillary","EndoMT",
                "Pericyte","SMC_1","SMC_2","SMC_3",
                "Fib_1","Fib_2","Fib_3","Fib_4","Fib_5","Fib_6")

clean_sub$Cell_type <- factor(clean_sub$Cell_type, levels = rev(my_order))

genes_oi = c("PCDH9",'GPM6A',"IL1RAPL1","PPP2R2B","MYH11","TAGLN","FLT1","CARMN","CEMIP","ROBO2","MOBP","SLC1A2","CLDN5","CLDN11","CTNNA3","CADM2","BMX","GJA5")  

## After decontX
DefaultAssay(clean_sub) = "decontX"
clean_sub@assays$decontX$counts = round(LayerData(clean_sub, assay = "decontX", layer = "counts"))

clean_sub = NormalizeData(clean_sub)
## plot the dot plot with color white to dark red
p3 = DotPlot(clean_sub,features = genes_oi,cols = c("white","darkred"),group.by = "Cell_type",scale = FALSE,assay = "decontX",) +
            RotatedAxis() + theme(axis.title = element_blank()) + ggtitle("After decontX_subsetted")

In [ ]:
options(repr.plot.width = 14,repr.plot.height = 7)

ggsave(patchwork::wrap_plots(p2,p3,ncol = 2), file = "./Results/Revision_2/Figures/After_decontX_dotplot.png",width = 15,height = 7)
ggsave(patchwork::wrap_plots(p2,p3,ncol = 2), file = "./Results/Revision_2/Figures/After_decontX_dotplot.pdf",width = 15,height = 7)

patchwork::wrap_plots(p2,p3,ncol = 2)